
# 07 常见因子与标签工程

> **本章要解决的问题**：同一个 5 日动量因子，为什么用 1 日收益算 IC 是正的、用 20 日收益算却是负的？期货的 carry、基差、期限结构，到底在赌什么？三障碍标签和 meta-label 比“未来 5 日收益”多了什么信息？标签错了，再好的因子也白搭。

## 本章导读

前六本 advanced 讲执行、数据、风险与运维；这一本回到研究本身，把“因子”与“标签”放在一起看。因子回答“用什么预测”，标签回答“预测什么”。两者必须成对设计：同一个因子，换一种标签，IC 可能从正变负。

本章分三块：

1. **因子目录**：期货/期权常用的九类因子——动量、carry/展期收益、基差、期限结构斜率与曲率、波动率、偏斜/风险逆转、持仓/成交/流动性、库存、价值/相对价差；
2. **标签工程**：固定期限远期收益、可成交价与执行延迟、roll-adjusted、三障碍、meta-label、路径标签（MFE/MAE）、期权标签（IV 变化/RV/delta-hedged PnL/carry）；
3. **标签的坑**：标签重叠、purge/embargo、`available_at`、标签库 schema 与注册表。

**注意**：本章所有数据都是合成数据，只用于演示方法；里面的 IC 数值不代表任何真实市场。

### 学习目标

- 给定一个期货/期权品种，能从九类因子里选出合适的候选，并写清经济逻辑、构造公式、数据字段、方向与适用期限。
- 给定价格序列，能构造 1 日/5 日/20 日固定期限远期收益，并说明可成交价与执行延迟如何改变标签。
- 给定波动率与时间窗口，能构造三障碍标签、meta-label 与 MFE/MAE 路径标签，并解释各自的用途与参数敏感性。
- 给定一份带披露时间的报表，能用 `available_at` 与 `merge_asof` 把数据对齐到“当时可见”，并识别按报告日对齐的前视。
- 给定一个因子与多个标签，能算 IC/ICIR 并解释“同一因子换标签结论不同”。
- 给定一组标签定义，能建 sqlite 标签注册表、算标签重叠、用 purge/embargo 切分样本，并输出可追溯的 JSON manifest。

### 先修

第 03 章的单因子报告与截面分组；第 30 章的收益归因与增量价值；第 38 章的版本与 manifest；第 40 章的 PIT 与双时间轴；会 pandas 的 `groupby`、`merge_asof`、`shift`。

## 在量化研究中的位置

- **谁用**：因子研究员、CTA/期权策略研究员、数据工程、组合经理。
- **输入**：期货多合约行情、现货/库存/持仓数据、期权隐波与成交、披露时间表。
- **输出**：候选因子清单、标签面板、IC/ICIR 表、标签注册表、可复现的 manifest。
- **在流程的哪一步**：从“读研报/想逻辑”到“单因子验证”之间；决定后面所有验证的口径。
- **失败模式**：因子和标签的时间尺度不匹配、用未来才可见的数据、标签重叠却当独立样本、只报一个标签的 IC。
- **最小工作流**：列因子目录 → 定标签 → 对齐 available_at → 算 IC/ICIR → 查重叠/purge → 登记标签库。

### 环境准备



In [1]:

import hashlib
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.2 | numpy 2.4.4



## 1. 常见因子目录（期货/期权方向）

期货与股票因子的最大区别是：**期货有到期日、有期限结构、有展期**。同一个“价格”，在不同合约、不同月份上含义不同；期权还多了一层波动率曲面。下面按“价格类 / 期限结构类 / 期权类 / 供需类”列一张最小目录。

### 1.1 价格类与期限结构类

| 因子 | 经济逻辑 | 构造/公式 | 数据字段 | 方向 | 适用期限 | 失效场景 |
|---|---|---|---|---|---|---|
| 动量 | 趋势延续 | $P_t/P_{t-h}-1$ | 近月收盘价 | 正 | 1–20 日 | 震荡市、换月跳空 |
| carry/展期收益 | 持有到期的展期盈亏 | $\approx \frac{P_{\text{near}}-P_{\text{mid}}}{P_{\text{mid}}}\cdot\frac{252}{\Delta}$ | 近/远月价差 | carry 高做多 | 1–3 月 | 期限结构突变、政策干预 |
| 基差 | 现货与期货的价差，反映库存与便利收益 | $(P_{\text{near}}-P_{\text{spot}})/P_{\text{spot}}$ | 现货、近月 | 看库存周期 | 1–20 日 | 现货数据质量、交割约束 |
| 期限结构斜率 | 曲线陡峭度，反映供需松紧 | $(\ln P_{\text{far}}-\ln P_{\text{near}})/\Delta\cdot252$ | 多合约 | 正斜率=contango | 1–3 月 | 近月逼仓、流动性断层 |
| 期限结构曲率 | 曲线弯曲程度，极端曲率易均值回归 | $(P_{\text{mid}}-P_{\text{near}})-(P_{\text{far}}-P_{\text{mid}})$ 归一 | 三合约 | 反向 | 1–2 周 | 合约不连续、成交稀疏 |

### 1.2 期权类与供需类

| 因子 | 经济逻辑 | 构造/公式 | 数据字段 | 方向 | 适用期限 | 失效场景 |
|---|---|---|---|---|---|---|
| 波动率 | 波动率风险溢价与均值回归 | $\operatorname{std}(r,20)\sqrt{252}$ | 收益率 | 低波偏好 | 1–20 日 | 波动率突变、结构断点 |
| 偏斜/风险逆转 | 尾部风险的定价 | $IV_{25\Delta put}-IV_{25\Delta call}$ | 期权链 IV | 极值反向 | 1–3 月 | 深虚值流动性差、IV 噪声 |
| 持仓/成交/流动性 | 资金关注度与拥挤度 | $OI_t/OI_{t-k}-1$ | 持仓量、成交量 | 视策略 | 1–5 日 | 主力换月、逼仓 |
| 库存 | 供需与便利收益 | $Inv_t/Inv_{t-k}-1$ | 库存周报 | 库存降→价格强 | 1–4 周 | 数据滞后、修订 |
| 价值/相对价差 | 跨品种比价均值回归 | $\ln(P_1/P_2)-\text{均值}$ | 两品种价格 | 反向 | 1–4 周 | 结构性变化、替代关系改变 |

**因子卡要写六件事**：经济逻辑（为什么有效）、公式（怎么算）、字段（从哪来）、方向（正/负）、适用期限（预测多长）、失效场景（什么时候不灵）。缺一项，后面的人就会用错。

### 1.3 代码：合成面板与六个因子

下面构造 6 个品种、120 个交易日的合成期货面板（含近/中/远月、现货、库存、持仓、隐波与偏斜），并计算 6 个因子。数据刻意做成“短期动量 + 长期反转”，方便后面对比不同标签下的 IC。



In [2]:

assets = ["CU", "RB", "M", "TA", "AU", "SC"]
T = 120
dates = pd.bdate_range("2024-01-02", periods=T)
N = len(assets)
rng = np.random.default_rng(4707)
mu = np.array([0.0003, -0.0001, 0.0002, 0.0, 0.0004, 0.0001])
sig = np.array([0.011, 0.013, 0.012, 0.014, 0.008, 0.015])

logp = np.zeros((T, N))
carry = np.zeros((T, N))
basis = np.zeros((T, N))
inv = np.zeros((T, N))
oi = np.zeros((T, N))
iv = np.zeros((T, N))
ivskew = np.zeros((T, N))
for i in range(N):
    logp[0, i] = 4.4 + rng.normal(0, 0.1)
    carry[0, i] = rng.normal(0.01, 0.02)
    basis[0, i] = rng.normal(0.0, 0.008)
    inv[0, i] = rng.normal(0, 1)
    oi[0, i] = rng.normal(11, 0.3)
    iv[0, i] = 0.22
    ivskew[0, i] = -0.03
    for t in range(1, T):
        mom5 = logp[t - 1, i] - logp[max(0, t - 5), i]
        gap20 = logp[t - 1, i] - logp[max(0, t - 20):t, i].mean()
        r = mu[i] + 0.20 * mom5 - 0.12 * gap20 + sig[i] * rng.normal()
        logp[t, i] = logp[t - 1, i] + r
        carry[t, i] = 0.97 * carry[t - 1, i] + rng.normal(0, 0.006)
        basis[t, i] = 0.95 * basis[t - 1, i] + rng.normal(0, 0.004)
        inv[t, i] = 0.97 * inv[t - 1, i] + rng.normal(0, 1)
        oi[t, i] = 0.98 * oi[t - 1, i] + rng.normal(0, 0.05)
        iv[t, i] = 0.9 * iv[t - 1, i] + 0.1 * (0.2 + 3.0 * sig[i]) + 0.01 * rng.normal()
        ivskew[t, i] = 0.9 * ivskew[t - 1, i] + rng.normal(0, 0.004)

spot = np.exp(logp)
near = spot * np.exp(basis)
mid = near * np.exp(-carry * 30 / 252)
far = near * np.exp(-carry * 90 / 252)

rows = []
for i, a in enumerate(assets):
    for t in range(T):
        rows.append({"date": dates[t], "asset": a, "spot": spot[t, i], "near": near[t, i],
                     "mid": mid[t, i], "far": far[t, i], "carry": carry[t, i],
                     "basis": basis[t, i], "inventory": inv[t, i], "oi": oi[t, i],
                     "iv_atm": iv[t, i], "iv_skew": ivskew[t, i]})
panel = pd.DataFrame(rows).sort_values(["asset", "date"]).reset_index(drop=True)
panel["ret"] = panel.groupby("asset")["near"].pct_change()
panel["mom5"] = panel.groupby("asset")["near"].pct_change(5)
panel["mom20"] = panel.groupby("asset")["near"].pct_change(20)
panel["vol20"] = panel.groupby("asset")["ret"].transform(lambda s: s.rolling(20).std() * np.sqrt(252))
panel["carry_f"] = panel["carry"] * 100
panel["basis_f"] = panel["basis"] * 100
panel["oi_chg5"] = panel.groupby("asset")["oi"].pct_change(5)
panel["iv_skew_f"] = panel["iv_skew"] * 100
print("面板行数", len(panel), "| 品种", panel["asset"].nunique(), "| 日期", panel["date"].nunique())
print(panel[["asset", "date", "near", "mom5", "vol20", "carry_f"]].dropna().head(4).to_string(index=False))


面板行数 720 | 品种 6 | 日期 120
asset       date      near      mom5    vol20   carry_f
   AU 2024-01-30 87.986101 -0.023085 0.159850 -1.285279
   AU 2024-01-31 86.744061 -0.037449 0.165924 -2.507179
   AU 2024-02-01 87.069708 -0.037733 0.166355 -2.508475
   AU 2024-02-02 87.317174 -0.027875 0.162715 -2.081799



**读数**：面板 720 行 = 6 品种 × 120 天。`mom5`、`carry_f`、`basis_f`、`vol20`、`oi_chg5`、`iv_skew_f` 六个因子都已经落到同一张长表里，索引是 `(asset, date)`。注意 `carry_f` 是把年化 carry 乘 100 后的百分点，`vol20` 是年化波动率，量纲不同但截面排序不受影响——**因子值只用来排序/标准化，绝对量级要在因子卡里写清**。

**常见错误**：

1. 不同合约的价格直接拼成一条序列算动量，换月日出现假跳空。
2. 把 carry 的方向搞反：contango（远月贵）时持有近月展期是亏钱的。
3. 因子值不做截面标准化就合成，量纲大的因子主导权重。

---



## 2. 标签工程：先定义“赚的是什么”

因子是“预测变量”，标签是“被预测的目标”。标签定义比因子更容易被忽视，却决定了整条研究的方向。

### 2.1 固定期限远期收益

最常见的标签是未来 $h$ 期的收益：

$$y_{t,h} = \frac{P_{t+h}}{P_t} - 1,$$

其中 $h$ 可以是 1 日、5 日、20 日（也常用 1 个月、3 个月）。要写清三件事：

1. **用哪个价格**：收盘价、次日开盘价、还是买一/卖一？研究可以用中间价，但**可成交价**要按方向取价差的不利一边；
2. **执行延迟**：信号在 $t$ 收盘产生，最早 $t+1$ 才能成交，标签的分母价格要用 $t+1$，不能再用 $t$；
3. **roll-adjusted**：期货要先把换月造成的假跳空处理掉，标签才是“持有合约的收益”，而不是“两个合约的价差”。

### 2.2 标签与因子必须同尺度

用 5 日动量因子去预测 20 日收益，IC 往往会被“时间尺度错配”稀释：因子捕捉的是短期信息，标签问的是中期结果。反过来，用 20 日因子配 1 日标签，等于让慢信号解释快噪声。**先想清楚要交易什么频率，再定标签的 $h$**。

### 2.3 代码：三种固定期限 + 三障碍 + meta + 路径标签



In [3]:

def add_labels(df):
    df = df.sort_values(["asset", "date"]).copy()
    for h in (1, 5, 20):
        df[f"fwd{h}"] = df.groupby("asset")["near"].transform(lambda s: s.shift(-h) / s - 1)
    trip = np.zeros(len(df))
    mfe = np.full(len(df), np.nan)
    mae = np.full(len(df), np.nan)
    for a in df["asset"].unique():
        idx = df.index[df["asset"] == a]
        sub = df.loc[idx].reset_index(drop=True)
        p = sub["near"].to_numpy()
        v = sub["vol20"].to_numpy()
        for t in range(len(sub) - 5):
            if np.isnan(v[t]):
                continue
            base = p[t]
            width = 0.8 * v[t] * np.sqrt(5 / 252) * base
            up, dn = base + width, base - width
            lab = 0
            for j in range(1, 6):
                if p[t + j] >= up:
                    lab = 1
                    break
                if p[t + j] <= dn:
                    lab = -1
                    break
            trip[idx[t]] = lab
            path = p[t + 1:t + 6] / base - 1
            mfe[idx[t]] = path.max()
            mae[idx[t]] = path.min()
    df["triple5"] = trip
    df["mfe5"] = mfe
    df["mae5"] = mae
    df["meta5"] = np.where(df["mom5"].isna() | df["fwd5"].isna(), np.nan,
                           np.where(np.sign(df["mom5"]) * df["fwd5"] > 0, 1.0, 0.0))
    df["rv_fwd5"] = df.groupby("asset")["ret"].transform(
        lambda s: s.shift(-1).rolling(5).std() * np.sqrt(252))
    df["iv_chg5"] = df.groupby("asset")["iv_atm"].transform(lambda s: s.shift(-5) - s)
    return df


panel = add_labels(panel)
print("标签缺失统计：", {c: int(panel[c].isna().sum()) for c in ["fwd1", "fwd5", "fwd20", "triple5", "meta5", "iv_chg5"]})
print("三障碍分布：", panel["triple5"].value_counts().to_dict())
print(panel[["asset", "date", "fwd1", "fwd5", "triple5", "meta5"]].dropna().head(4).to_string(index=False))


标签缺失统计： {'fwd1': 6, 'fwd5': 30, 'fwd20': 120, 'triple5': 0, 'meta5': 60, 'iv_chg5': 30}
三障碍分布： {0.0: 347, 1.0: 214, -1.0: 159}
asset       date      fwd1      fwd5  triple5  meta5
   AU 2024-01-09 -0.019808 -0.019958      0.0    0.0
   AU 2024-01-10  0.008838  0.012713      0.0    0.0
   AU 2024-01-11  0.007606 -0.010687      0.0    1.0
   AU 2024-01-12 -0.014660 -0.011306      0.0    1.0



**读数**：720 行里，`fwd1` 缺 6 行（每品种最后 1 天）、`fwd5` 缺 30 行、`fwd20` 缺 120 行（每品种最后 20 天）——这是**标签天然会吃掉数据尾部**，不是数据错误。三障碍 5 日标签分三类：向上触障 214 行、向下触障 159 行、时间障碍 347 行，类别不平衡很明显。`meta5` 缺 60 行（因为 `mom5` 前 5 天为空）。这些缺失会在算 IC 时被自动剔除，但**必须在标签库的覆盖率里写清楚**。

**常见错误**：

1. 用最后 20 天做回测却不留出标签期，等于用未来的标签训练。
2. 三障碍的障碍宽度忘了乘 $\sqrt{h}$，5 日障碍和 1 日一样宽。
3. meta-label 直接用未来收益的正负，而不先固定主信号方向。

---

## 3. 进阶标签：三障碍、meta-label、路径与期权

### 3.1 三障碍标签

固定期限标签只问“$h$ 期后涨跌多少”，三障碍还问“**先碰到哪个障碍**”。设 5 日窗口、上下障碍为 $\pm k\sigma\sqrt{h}$、时间障碍为 $h$：

| 结果 | 含义 | 标签 |
|---|---|---|
| 先碰到上障碍 | 在窗口内上涨超过阈值 | +1 |
| 先碰到下障碍 | 在窗口内下跌超过阈值 | −1 |
| 到窗口末都没碰到 | 横盘/震荡 | 0 |

它同时编码了方向、幅度和时间，适合做止损/止盈类策略；代价是对 $k$、$h$ 敏感，且类别容易不平衡（本例时间障碍占 48%）。实盘还要考虑**盘中触及还是收盘确认**——用收盘价会低估触障频率。

### 3.2 meta-label 与路径标签

- **meta-label**：先有一个方向信号（主模型），标签只回答“这个方向对不对”（0/1）。优点是不必重复学习方向，只学“该不该下注、下多重”；常配合仓位缩放。
- **路径标签 MFE/MAE**：窗口内最大有利偏移（Maximum Favorable Excursion）与最大不利偏移（Maximum Adverse Excursion）。它不直接当标签，而是帮你标定止损、止盈和仓位——比如“MAE 超过 −3% 的单子后来大多亏损”，止损线就有依据。

### 3.3 期权标签

期权策略的标签要落到“波动率与对冲后的盈亏”上，而不是标的价格：

| 标签 | 预测目标 | 公式/构造 | 用途 | 陷阱 |
|---|---|---|---|---|
| IV 变化 | 隐波方向 | $\Delta IV = IV_{t+h}-IV_t$ | 波动率交易 | 深虚值 IV 噪声大 |
| RV | 已实现波动 | $\operatorname{std}(r_{t+1:t+h})\sqrt{252}$ | 波动率溢价 | 采样频率敏感 |
| delta-hedged PnL | 对冲后的真实盈亏 | $\Delta V - \Delta\cdot\Delta S$ | 期权策略评估 | 依赖对冲频率与成本 |
| carry | 时间价值+基差 | $\Theta\Delta t$ + 展期收益 | 卖方策略 | 尾部风险被低估 |

### 3.4 available_at：数据什么时候“可用”

报表有 `report_date`（统计截止日），但真正能用于交易的是 `available_at`（披露/入库后可见的时间）。库存周报周五统计、下周一才公布；如果用 `report_date` 对齐，等于周五就知道周一的数据，典型前视。正确做法是用 `merge_asof(direction="backward")` 按 `available_at` 对齐。

### 3.5 代码：库存周报的 available_at 对齐



In [4]:

inv_rows = []
for i, a in enumerate(assets):
    sub = panel[panel["asset"] == a].reset_index(drop=True)
    for t in range(0, T, 5):
        inv_rows.append({"asset": a, "report_date": dates[t], "value": sub.loc[t, "inventory"]})
inv_report = pd.DataFrame(inv_rows)
inv_report["available_at"] = inv_report["report_date"] + pd.offsets.BDay(1)

panel = panel.sort_values(["asset", "date"]).reset_index(drop=True)
asof_parts, naive_parts = [], []
for a in assets:
    left = panel.loc[panel["asset"] == a, ["date"]].sort_values("date")
    right = inv_report.loc[inv_report["asset"] == a, ["available_at", "value"]].sort_values("available_at")
    m = pd.merge_asof(left, right, left_on="date", right_on="available_at", direction="backward")
    asof_parts.append(m["value"])
    right2 = inv_report.loc[inv_report["asset"] == a, ["report_date", "value"]].sort_values("report_date")
    m2 = pd.merge_asof(left, right2, left_on="date", right_on="report_date", direction="backward")
    naive_parts.append(m2["value"])
panel["inv_asof"] = pd.concat(asof_parts).sort_index().to_numpy()
panel["inv_naive"] = pd.concat(naive_parts).sort_index().to_numpy()
panel["inv_chg5"] = panel.groupby("asset")["inv_asof"].pct_change(5)
diff = (panel["inv_asof"] - panel["inv_naive"]).abs()
print("库存周报", len(inv_report), "条 | 首条 report_date", inv_report.loc[0, "report_date"].date(),
      "available_at", inv_report.loc[0, "available_at"].date())
print("as-of 对齐覆盖", int(panel["inv_asof"].notna().sum()), "行 | 与按报告日对齐不同",
      int((diff > 1e-12).sum()), "行 | 平均绝对差", round(float(diff.mean()), 4))
sample = panel[(panel["asset"] == "CU") & (panel["date"] >= dates[2]) & (panel["date"] <= dates[12])]
print(sample[["date", "inv_asof", "inv_naive", "inv_chg5"]].to_string(index=False))


库存周报 144 条 | 首条 report_date 2024-01-02 available_at 2024-01-03
as-of 对齐覆盖 714 行 | 与按报告日对齐不同 138 行 | 平均绝对差 0.3432
      date  inv_asof  inv_naive  inv_chg5
2024-01-04 -0.720635  -1.612052       NaN
2024-01-05  0.635846  -1.070900       NaN
2024-01-08 -3.168886  -0.959487       NaN
2024-01-09 -4.951927  -7.064300  3.327054
2024-01-10 -7.064300  -7.064300  0.498163
2024-01-11 -1.612052  -1.612052  1.236987
2024-01-12 -5.614625  -5.614625 -9.830161
2024-01-15 -1.070900  -1.070900 -0.662058
2024-01-16 -0.959487  -0.959487 -0.806240
2024-01-17  1.651728   1.651728 -1.233813
2024-01-18  1.651728   1.651728 -2.024612



**读数**：库存周报 144 条（6 品种 × 24 期），首条 `report_date` 是 1 月 2 日、`available_at` 是 1 月 3 日。按 `available_at` 对齐后有 714 行拿到库存值；和“按报告日对齐”相比，有 138 行不同，平均绝对差 0.3432——这 138 行就是前视污染的规模。看样本：1 月 4 日 as-of 值是 −0.72（用的是 1 月 2 日发布、1 月 3 日可见的那期），而按报告日对齐已经用上了 1 月 4 日当天的报告。

**常见错误**：

1. 用 `report_date` 或自然日对齐，把还没公布的数据提前使用。
2. `merge_asof` 没按 `available_at` 排序，结果静默错误。
3. 库存这类低频数据不按周对齐，日频面板里出现“同一条库存被重复使用 5 天”却不知情。

---



## 5. 同一因子，不同标签：IC 会变

### 5.1 怎么算 IC

对每个交易日做**截面**秩相关：

$$IC_t = \operatorname{Spearman}\big(f_{t}, y_{t}\big),$$

再对 $t$ 求均值与标准差，$ICIR = \overline{IC}/\sigma_{IC}$。注意：样本里只有 6 个品种，截面 IC 的噪声很大（单期标准差约 0.45），所以均值要结合期数和 ICIR 一起看，不能只看一个数。

### 5.2 代码：固定因子，换标签



In [5]:

def spearman(a, b):
    ra = pd.Series(a).rank()
    rb = pd.Series(b).rank()
    sa, sb = ra.std(ddof=1), rb.std(ddof=1)
    if not np.isfinite(sa) or not np.isfinite(sb) or sa == 0 or sb == 0:
        return np.nan
    return float(((ra - ra.mean()) * (rb - rb.mean())).sum() / (len(ra) * sa * sb))


def rank_ic_panel(df, factor, label):
    sub = df.dropna(subset=[factor, label])
    ics = []
    for d, g in sub.groupby("date"):
        if len(g) < 5:
            continue
        ic = spearman(g[factor], g[label])
        if np.isfinite(ic):
            ics.append(ic)
    ics = np.array(ics)
    if len(ics) == 0:
        return np.nan, np.nan, 0
    sd = ics.std(ddof=1)
    return ics.mean(), (ics.mean() / sd if sd > 0 else np.nan), len(ics)


label_names = {"fwd1": "1日收益", "fwd5": "5日收益", "fwd20": "20日收益",
               "triple5": "三障碍5日", "meta5": "元标签5日", "iv_chg5": "IV变化5日"}
rows = []
for lab, cname in label_names.items():
    ic, icir, n = rank_ic_panel(panel, "mom5", lab)
    rows.append({"标签": cname, "IC": round(ic, 3), "ICIR": round(icir, 2), "期数": n})
print("同一个 5 日动量因子，换标签后的 IC：")
print(pd.DataFrame(rows).to_string(index=False))

factors = ["mom5", "mom20", "carry_f", "basis_f", "vol20", "iv_skew_f"]
mat = pd.DataFrame(index=factors, columns=list(label_names.values()), dtype=float)
for f in factors:
    for lab, cname in label_names.items():
        mat.loc[f, cname] = rank_ic_panel(panel, f, lab)[0]
print("因子 × 标签 IC 矩阵：")
print(mat.round(3).to_string())


同一个 5 日动量因子，换标签后的 IC：
    标签     IC  ICIR  期数
  1日收益  0.133  0.35 114
  5日收益  0.013  0.03 110
 20日收益 -0.301 -0.86  95
 三障碍5日  0.039  0.10  95
 元标签5日  0.023  0.06 107
IV变化5日  0.016  0.04 110


因子 × 标签 IC 矩阵：
            1日收益   5日收益  20日收益  三障碍5日  元标签5日  IV变化5日
mom5       0.133  0.013 -0.301  0.039  0.023   0.016
mom20     -0.143 -0.336 -0.250 -0.273  0.021   0.116
carry_f   -0.025 -0.021 -0.138 -0.030 -0.070   0.008
basis_f   -0.057 -0.092 -0.142 -0.015  0.020   0.073
vol20      0.018  0.019  0.053 -0.017  0.039  -0.042
iv_skew_f -0.004  0.079 -0.008  0.024  0.058  -0.035



**读数**：同一个 `mom5`，换标签后 IC 从 +0.133（1 日）掉到 +0.013（5 日），再到 −0.301（20 日）——**符号直接翻转**。原因是这份合成数据里“短期动量 + 长期反转”同时存在：5 日动量对下一日收益有延续性，但对 20 日收益反而是反向指标。三障碍与 meta-label 的 IC 只有 0.039 和 0.023：它们把连续收益压成离散类别，信息被压缩，同一因子在离散标签上看起来“变弱”了。因子 × 标签矩阵还显示：`mom20` 在所有收益类标签上都是负的（−0.143/−0.336/−0.250），`carry_f`、`basis_f` 只在 20 日标签上明显（−0.138/−0.142）。

**结论：IC 不是因子的固有属性，而是“因子 + 标签 + 样本”三者的联合结果。** 报告 IC 时必须写清标签定义与期限，跨标签的 IC 不可直接比较。

**常见错误**：

1. 只报一个标签的 IC，却把它当成因子“有没有用”的最终结论。
2. 用 1 日标签筛因子、用 20 日标签做交易，筛选与执行口径不一致。
3. 三障碍/meta 标签的 IC 天然低于连续收益标签，却拿同一把尺子比高低。

---

## 6. 标签库：重叠、purge/embargo 与注册表

### 6.1 标签重叠：独立样本没那么多

$h$ 日标签的相邻两天共享 $h-1$ 天收益窗口，重叠比例是 $(h-1)/h$：5 日标签重合约 80%，20 日标签重合约 95%。这会让 IC 的统计显著性被高估——名义上 100 期，有效独立样本可能只有 100/h 期。所以标签期限越长，越要用 block bootstrap 或按重叠期修正 t 值。

### 6.2 purge 与 embargo

时间序列交叉验证不能简单切分：训练集最后 $h$ 天的标签窗口会伸进测试集，等于“训练时看到了测试期的价格”。处理办法：

1. **purge**：把训练集末尾 $h$ 天删掉；
2. **embargo**：在测试集之后、下一段训练之前再留几天缓冲，避免自相关泄漏。

### 6.3 标签库 schema 与注册表

标签定义也要版本化。最小 schema 包含：`label_id`、名称、类型（forward_return/triple_barrier/meta_label/option）、期限 $h$、是否依赖 `available_at`、描述、版本。存进数据库后导出 JSON manifest，附覆盖率与 hash，任何一份研究都能追溯到“用的是哪版标签”。

### 6.4 代码：重叠、purge/embargo 与 sqlite 注册表



In [6]:

for h in (5, 20):
    print(f"horizon={h}：相邻标签收益窗口重叠比例 {(h - 1) / h:.2f}，有效独立样本约为行数的 1/{h}")
train = np.arange(0, 80)
test = np.arange(80, 100)
h = 20
purged = train[train < test[0] - h]
print("训练 80 天 -> purge 掉末尾", 80 - len(purged), "天后剩", len(purged), "天")
next_block = np.arange(100, 120)
embargoed = next_block[5:]
print("测试后下一段训练", len(next_block), "天 -> embargo 5 天后剩", len(embargoed), "天")

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""
CREATE TABLE label_registry (
    label_id TEXT PRIMARY KEY, name TEXT NOT NULL, kind TEXT NOT NULL,
    horizon INTEGER NOT NULL, uses_available_at INTEGER NOT NULL,
    description TEXT NOT NULL, version TEXT NOT NULL
)
""")
label_rows = [
    ("L-fwd1", "固定期限1日", "forward_return", 1, 1, "次一根 K 线收盘到收盘", "v1"),
    ("L-fwd5", "固定期限5日", "forward_return", 5, 1, "未来 5 日累计收益", "v1"),
    ("L-fwd20", "固定期限20日", "forward_return", 20, 1, "未来 20 日累计收益", "v1"),
    ("L-triple5", "三障碍5日", "triple_barrier", 5, 1, "上下障碍 ±0.8σ5、时间障碍 5 日", "v1"),
    ("L-meta5", "元标签5日", "meta_label", 5, 1, "主信号方向是否正确的二分类标签", "v1"),
    ("L-ivchg5", "IV变化5日", "option", 5, 1, "平值隐波 5 日变化", "v1"),
]
cur.executemany("INSERT INTO label_registry VALUES (?,?,?,?,?,?,?)", label_rows)
conn.commit()
print("注册表记录：", cur.execute("SELECT label_id, kind, horizon FROM label_registry ORDER BY label_id").fetchall())

cov = {"L-fwd1": "fwd1", "L-fwd5": "fwd5", "L-fwd20": "fwd20", "L-triple5": "triple5",
       "L-meta5": "meta5", "L-ivchg5": "iv_chg5"}
coverage = {k: int(panel[v].notna().sum()) for k, v in cov.items()}
loc = panel["triple5"].value_counts().to_dict()
manifest = {
    "schema": ["label_id", "name", "kind", "horizon", "uses_available_at", "description", "version"],
    "labels": [dict(zip(["label_id", "name", "kind", "horizon", "uses_available_at", "description", "version"], r)) for r in label_rows],
    "coverage": coverage,
    "triple_balance": {str(k): int(v) for k, v in loc.items()},
}
manifest_json = json.dumps(manifest, ensure_ascii=False, sort_keys=True)
print("标签覆盖：", coverage)
print("三障碍类别分布：", manifest["triple_balance"])
print("注册表 hash：", hashlib.sha256(manifest_json.encode()).hexdigest()[:12])


horizon=5：相邻标签收益窗口重叠比例 0.80，有效独立样本约为行数的 1/5
horizon=20：相邻标签收益窗口重叠比例 0.95，有效独立样本约为行数的 1/20
训练 80 天 -> purge 掉末尾 20 天后剩 60 天
测试后下一段训练 20 天 -> embargo 5 天后剩 15 天
注册表记录： [('L-fwd1', 'forward_return', 1), ('L-fwd20', 'forward_return', 20), ('L-fwd5', 'forward_return', 5), ('L-ivchg5', 'option', 5), ('L-meta5', 'meta_label', 5), ('L-triple5', 'triple_barrier', 5)]
标签覆盖： {'L-fwd1': 714, 'L-fwd5': 690, 'L-fwd20': 600, 'L-triple5': 720, 'L-meta5': 660, 'L-ivchg5': 690}
三障碍类别分布： {'0.0': 347, '1.0': 214, '-1.0': 159}
注册表 hash： 0f4a7014a965



**读数**：5 日标签相邻重叠 80%、20 日标签重叠 95%，后者的有效独立样本只有约 1/20。训练 80 天做 20 日标签验证，purge 掉末尾 20 天后只剩 60 天；测试之后的下一段训练 20 天，embargo 5 天后剩 15 天。注册表 6 条标签记录都能按 `label_id` 查到类型与期限；标签覆盖率从 600（20 日）到 720（三障碍）不等——三障碍没有尾部缺失，但对前 20 天（`vol20` 未成形）会留 0 类别，需要在训练时单独处理。JSON manifest 的 hash 是 `0f4a7014a965`，把它写进实验记录，就能回答“这次研究用的是哪版标签”。

**常见错误**：

1. 把重叠标签当独立样本，t 值虚高。
2. 只 purge 不 embargo，或者反过来；两者解决的是不同方向的泄漏。
3. 标签定义改了却复用旧的 `label_id` 与版本号，历史结果无法复现。

---

## 本章小结

### 一张表

| 主题 | 关键问题 | 本章做法 |
|---|---|---|
| 因子目录 | 用什么预测 | 九类因子：经济逻辑/公式/字段/方向/期限/失效 |
| 固定期限标签 | 预测多长 | 1/5/20 日远期收益 + 可成交价 + roll-adjusted |
| 三障碍标签 | 方向+幅度+时间 | ±kσ√h 上下障碍、h 时间障碍，输出 +1/0/−1 |
| meta-label | 该不该下注 | 主信号方向已知，只学方向是否正确（0/1） |
| 路径标签 | 止损/止盈 | MFE/MAE 衡量窗口内最大有利/不利偏移 |
| 期权标签 | 波动率而非价格 | IV 变化、RV、delta-hedged PnL、carry |
| available_at | 什么时候可见 | merge_asof 按披露日对齐，杜绝前视 |
| 标签库 | 可复现 | 重叠/purge/embargo + sqlite 注册表 + JSON hash |

### 八句话

1. 因子回答“用什么预测”，标签回答“预测什么”。
2. 标签期限必须和交易频率匹配。
3. 固定期限标签要写清价格、执行延迟与换月处理。
4. 三障碍标签同时编码方向、幅度与时间。
5. meta-label 只学“该不该下注”，不重复学方向。
6. 数据要用 `available_at` 对齐，不能用统计日。
7. 标签重叠会高估显著性，purge/embargo 是切分的第一步。
8. 同一因子换标签，IC 可能从正变负——报 IC 必须报标签。

---

## 本章习题

**1.** 期货 carry 因子怎么构造？contango 时做多近月展期的收益是正还是负？

**2.** 基差因子反映什么？为什么说它和库存、便利收益有关？

**3.** 期限结构“斜率”和“曲率”各抓什么信号？分别适合多长周期？

**4.** 固定期限标签要写清哪三件事？

**5.** 三障碍标签比固定期限标签多了什么信息？它对哪些参数敏感？

**6.** meta-label 与主信号的分工是什么？为什么它能减少重复学习？

**7.** `report_date` 与 `available_at` 有什么区别？用错会导致什么？

**8.** 20 日标签的 IC 看起来更“稳”，但有效独立样本为什么反而更少？该怎么处理？

<details><summary>参考答案</summary>

**1.** carry ≈（近月−远月）/远月 × 年化因子，反映持有近月到期后展期到远月的盈亏；contango 时远月更贵，做多近月并展期是负 carry（展期亏损）。

**2.** 基差是现货与期货的价差，反映库存松紧与便利收益：库存低、现货紧张时基差走强；库存高、现货宽松时基差走弱。

**3.** 斜率抓曲线整体陡峭度（供需松紧/contango 程度），适合 1–3 个月；曲率抓曲线弯曲/极端程度（近月逼仓、局部紧张），回归更快，适合 1–2 周。

**4.** 用哪个价格（收盘/开盘/买卖价）、执行延迟（信号日到成交日）、换月处理（roll-adjusted），三者共同决定标签口径。

**5.** 多了“先碰到哪个障碍”的路径信息，能区分“慢慢涨到目标”和“先暴跌再涨回”；对障碍宽度 k、窗口 h、以及判定方式（盘中触及还是收盘确认）敏感。

**6.** 主信号负责方向（做多还是做空），meta-label 只回答“这个方向对不对、该不该下注、下多重”；因为方向已由主信号给定，meta 模型只学二分类，避免重复学习方向。

**7.** `report_date` 是数据统计截止日，`available_at` 是数据真正可见/可用的时间；用 `report_date` 对齐会提前使用未公布数据，造成前视偏差。

**8.** 20 日标签的相邻窗口重叠 95%，名义样本虽多但有效独立样本只有约 1/20，IC 的噪声被低估；应使用 purge/embargo 切分、block bootstrap 或按重叠期修正标准误。

</details>

---

## 参考资料

- López de Prado, M. *Advances in Financial Machine Learning*（三障碍标签、meta-label、purge/embargo）。
- Grinold, R. & Kahn, R. *Active Portfolio Management*（IC/ICIR 与信息比率）。
- 上海期货交易所、大连商品交易所、郑州商品交易所合约与库存周报说明（品种细则以交易所官网为准）。
- 第 03 章（单因子报告与截面分组）；第 28 章（库存修订与当时可见的数据）；第 40 章（双时间轴与 PIT）。
